In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import networkx as nx

project_dir = Path(r"c:/msc_2/onlab2/Disease-Gene-Prioritization-Project")
data_dir = Path(project_dir, "data")
raw_data_dir = Path(data_dir, "raw_data")

ppi_path = Path(raw_data_dir, "9606.protein.links.detailed.v11.5.txt")

# Load interactions
print("Loading PPI table (this may take a moment)...")
ppi_df = pd.read_csv(ppi_path, sep=" ")
print(f"Raw interactions: {len(ppi_df):,}")

# Basic filter on score (adjustable)
min_score = 700  # 0-1000 (STRING); pick moderate-high confidence
ppi_df = ppi_df[ppi_df["combined_score"] >= min_score].copy()
print(f"Filtered interactions (score >= {min_score}): {len(ppi_df):,}")

# Extract ENSP ids (STRING ids look like "9606.ENSP00000...")
def strip_prefix(pid):
    return pid.split(".",1)[1] if "." in pid else pid

ppi_df["ensp1"] = ppi_df["protein1"].astype(str).map(strip_prefix)
ppi_df["ensp2"] = ppi_df["protein2"].astype(str).map(strip_prefix)

unique_ensps = pd.Index(ppi_df["ensp1"].tolist() + ppi_df["ensp2"].tolist()).unique()
print(f"{len(unique_ensps):,} unique ENSP ids found.")

Loading PPI table (this may take a moment)...
Raw interactions: 11,938,498
Filtered interactions (score >= 700): 505,968
16,814 unique ENSP ids found.


In [ ]:
import mygene
# import mydisease

mg = mygene.MyGeneInfo()
unique_ensps = pd.Index(ppi_df["ensp1"].tolist() + ppi_df["ensp2"].tolist()).unique()
print(f"Querying mygene for {len(unique_ensps):,} ENSP ids (may take ~1-2 min)...")
# mygene supports scopes='ensembl.protein'
query_batches = [unique_ensps[i:i+1000] for i in range(0, len(unique_ensps), 1000)]
ensp_to_symbol = {}
for batch in query_batches:
    res = mg.querymany(list(batch), scopes="ensembl.protein", fields="symbol", species="human", as_dataframe=False)
    for r in res:
        q = r.get('query')
        sym = r.get('symbol')
        if q and sym:
            ensp_to_symbol[q] = sym

print(f"Mapped ENSP -> gene symbol for {len(ensp_to_symbol):,} ids")

# Map back to dataframe
ppi_df["gene1"] = ppi_df["ensp1"].map(ensp_to_symbol)
ppi_df["gene2"] = ppi_df["ensp2"].map(ensp_to_symbol)

# Drop interactions where mapping failed or self-interactions
ppi_df = ppi_df.dropna(subset=["gene1","gene2"])
ppi_df = ppi_df[ppi_df["gene1"] != ppi_df["gene2"]]

# Collapse multiple edges between the same gene pair by taking max score
ppi_df["gA"] = ppi_df[["gene1","gene2"]].min(axis=1)
ppi_df["gB"] = ppi_df[["gene1","gene2"]].max(axis=1)
agg = ppi_df.groupby(["gA","gB"])["combined_score"].max().reset_index()

# Build NetworkX graph
G = nx.Graph()
for _, row in agg.iterrows():
    a, b, s = row["gA"], row["gB"], row["combined_score"]
    G.add_edge(a, b, weight=float(s)/1000.0)  # normalize to 0-1

print(f"Graph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

# Basic stats
deg = dict(G.degree())
top_deg = sorted(deg.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("Top nodes by degree:", top_deg)

# Save outputs
out_gexf = data_dir / "string_human_gene_graph.gexf"
out_edgelist = data_dir / "string_human_gene_edgelist.tsv"
nx.write_gexf(G, out_gexf)
nx.write_weighted_edgelist(G, out_edgelist, delimiter="\t")
print(f"Saved graph to {out_gexf} and {out_edgelist}")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Querying mygene for 16,814 ENSP ids (may take ~1-2 min)...


25 input query terms found no hit:	['ENSP00000035383', 'ENSP00000062104', 'ENSP00000074304', 'ENSP00000155858', 'ENSP00000204615', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
11 input query terms found no hit:	['ENSP00000236166', 'ENSP00000236918', 'ENSP00000238855', 'ENSP00000245323', 'ENSP00000246554', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
17 input query terms found no hit:	['ENSP00000259806', 'ENSP00000260645', 'ENSP00000261517', 'ENSP00000262888', 'ENSP00000263642', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
21 input query terms found no hit:	['ENSP00000267859', 'ENSP00000268035', 'ENSP00000269349', 'ENSP00000270279', 'ENSP00000272643', 'ENS


Mapped ENSP -> gene symbol for 16,129 ids
Graph built: 16,047 nodes, 235,252 edges
Top nodes by degree: [('RPS27A', 718), ('TP53', 699), ('UBA52', 646), ('UBC', 605), ('CTNNB1', 549), ('SRC', 546), ('UBB', 525), ('EP300', 490), ('AKT1', 464), ('EGFR', 456)]
Saved graph to c:\msc_2\onlab2\Disease-Gene-Prioritization-Project\data\string_human_gene_graph.gexf and c:\msc_2\onlab2\Disease-Gene-Prioritization-Project\data\string_human_gene_edgelist.tsv


In [5]:
res

[{'query': 'ENSP00000469421', 'notfound': True},
 {'query': 'ENSP00000469441',
  '_id': '246100',
  '_score': 29.85333,
  'symbol': 'CTAG1A'},
 {'query': 'ENSP00000469455',
  '_id': '4909',
  '_score': 29.85333,
  'symbol': 'NTF4'},
 {'query': 'ENSP00000469468',
  '_id': '25804',
  '_score': 29.85333,
  'symbol': 'LSM4'},
 {'query': 'ENSP00000469591',
  '_id': '89887',
  '_score': 29.85333,
  'symbol': 'ZNF628'},
 {'query': 'ENSP00000469613',
  '_id': '2323',
  '_score': 29.85333,
  'symbol': 'FLT3LG'},
 {'query': 'ENSP00000469647',
  '_id': '7538',
  '_score': 29.85333,
  'symbol': 'ZFP36'},
 {'query': 'ENSP00000469686',
  '_id': '112398',
  '_score': 29.85333,
  'symbol': 'EGLN2'},
 {'query': 'ENSP00000469689',
  '_id': '22941',
  '_score': 29.85333,
  'symbol': 'SHANK2'},
 {'query': 'ENSP00000469705', 'notfound': True},
 {'query': 'ENSP00000469759',
  '_id': '345193',
  '_score': 29.85333,
  'symbol': 'LRIT3'},
 {'query': 'ENSP00000469836',
  '_id': '84769',
  '_score': 29.85333,
  

In [4]:
ensp_to_symbol

{'ENSP00000000233': 'ARF5',
 'ENSP00000000412': 'M6PR',
 'ENSP00000001008': 'FKBP4',
 'ENSP00000001146': 'CYP26B1',
 'ENSP00000002125': 'NDUFAF7',
 'ENSP00000002165': 'FUCA2',
 'ENSP00000002596': 'HS3ST1',
 'ENSP00000002829': 'SEMA3F',
 'ENSP00000003084': 'CFTR',
 'ENSP00000003100': 'CYP51A1',
 'ENSP00000003302': 'USP28',
 'ENSP00000004531': 'SLC7A2',
 'ENSP00000005178': 'PDK4',
 'ENSP00000005226': 'USH1C',
 'ENSP00000005257': 'RALA',
 'ENSP00000005260': 'BAIAP2L1',
 'ENSP00000005284': 'CACNG3',
 'ENSP00000005286': 'TMEM132A',
 'ENSP00000005340': 'DVL2',
 'ENSP00000005386': 'RPAP3',
 'ENSP00000005587': 'SKAP2',
 'ENSP00000006015': 'HOXA11',
 'ENSP00000006053': 'CX3CL1',
 'ENSP00000006275': 'TRAPPC6A',
 'ENSP00000006658': 'SPATA20',
 'ENSP00000006777': 'RHBDD2',
 'ENSP00000007390': 'TSR3',
 'ENSP00000007414': 'OSBPL7',
 'ENSP00000007699': 'YBX2',
 'ENSP00000007722': 'ITGA3',
 'ENSP00000008391': 'TFAP2D',
 'ENSP00000008527': 'CRY1',
 'ENSP00000008938': 'PGLYRP1',
 'ENSP00000009041': 'STA

In [1]:
from pathlib import Path
import pandas as pd
import networkx as nx

project_dir = Path(r"c:/msc_2/onlab2/Disease-Gene-Prioritization-Project")
data_dir = project_dir / "data"
raw_dir = data_dir / "raw_data"

genes_to_diseases_path = Path(data_dir, "genes_to_disease.txt")
genes_to_phenotypes_path = Path(data_dir, "genes_to_phenotype.txt")
phenotype_path = Path(data_dir, "phenotype.hpoa")

genes_to_diseases_df = pd.read_csv(genes_to_diseases_path, sep="\t")
genes_to_phenotypes_df = pd.read_csv(genes_to_phenotypes_path, sep="\t")
phenotype_df = pd.read_csv(phenotype_path, sep="\t", comment="#")

C:\Users\wittd\AppData\Local\Temp\ipykernel_1208\2547921955.py:15: DtypeWarning: Columns (2,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  phenotype_df = pd.read_csv(phenotype_path, sep="\t", comment="#")


In [3]:
phenotype_df.head()

,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [ ]:
import requests
import json
import time
import pandas as pd

# Provide your API key
API_KEY = ""

# Specify query parameters by means of a dictionary 
params = {}
# ...retrieve disease associated to gene with NCBI ID equal to 351 
params["gene_ncbi_id"] = "351"
# ...retrieve the first page of results (page number 0) 
params["page_number"] = "0"

# Create a dictionary with the HTTP headers of your API call 
HTTPheadersDict = {}
# Set the 'Authorization' HTTP header equal to API_KEY (your API key) 
HTTPheadersDict['Authorization'] = API_KEY
# Set the 'accept' HTTP header to specify the response format: one of 'application/json', 'application/xml', 'application/csv' 
HTTPheadersDict['accept'] = 'application/json'

# Query the gda summary endpoint 
response = requests.get("https://api.disgenet.com/api/v1/gda/summary",\
                        params=params, headers=HTTPheadersDict, verify=False)

# If the status code of response is 429, it means you have reached one of your query limits 
# You can retrieve the time you need to wait until doing a new query in the response headers 
if not response.ok:
    if response.status_code == 429:
        while response.ok is False:
            print("You have reached a query limit for your user. Please wait {} seconds until next query".format(\
            response.headers['x-rate-limit-retry-after-seconds']))
            time.sleep(int(response.headers['x-rate-limit-retry-after-seconds']))
            print("Your rate limit is now restored")

            # Repeat your query
            response = requests.get("https://api.disgenet.com/api/v1/gda/summary",\
                                    params=params, headers=HTTPheadersDict, verify=False)
            if response.ok is True:
                break
            else:
                continue

# Parse response content in JSON format since we set 'accept:application/json' as HTTP header 
response_parsed = json.loads(response.text)
print('STATUS: {}'.format(response_parsed["status"]))
print('TOTAL NUMBER OF RESULTS: {}'.format(response_parsed["paging"]["totalElements"]))
print('NUMBER OF RESULTS RETRIEVED BY CURRENT CALL (PAGE NUMBER {}): {}'.format(\
      response_parsed["paging"]["currentPageNumber"], response_parsed["paging"]["totalElementsInPage"]))

c:\Users\wittd\anaconda3\envs\Disease-Gene-Prioritization-Project\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.disgenet.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


STATUS: OK
TOTAL NUMBER OF RESULTS: 57
NUMBER OF RESULTS RETRIEVED BY CURRENT CALL (PAGE NUMBER 0): 57


In [2]:
response_parsed

{'status': 'OK',
 'paging': {'pageSize': 100,
  'totalElements': 57,
  'totalElementsInPage': 57,
  'currentPageNumber': 0},
 'warnings': ['Academic roles can only access to CURATED sources: (CLINGEN, CLINVAR, GENCC, MGD_HUMAN, ORPHANET, PSYGENET, RGD_HUMAN, UNIPROT)',
  'gene_ensembl_id > The parameter value is empty / unspecified.',
  'gene_symbol > The parameter value is empty / unspecified.',
  'uniprot_id > The parameter value is empty / unspecified.',
  'disease > The parameter value is empty / unspecified.',
  'chemical_id > The parameter value is empty / unspecified.',
  'chemical_association > The parameter value is empty / unspecified.',
  'output > The parameter value is empty / unspecified.',
  'source > The parameter value is empty / unspecified.',
  'dis_type > The parameter value is empty / unspecified.',
  'dis_class_list > The parameter value is empty / unspecified.',
  'order_by > The parameter value is empty / unspecified.',
  'disease_inheritance > The parameter val